# Customer Migration to OneBill — Simplified Pipeline

End-to-end migration of accounts and their associated contacts into OneBill.

## Sources

| Data | Source | Purpose in payload |
|---|---|---|
| **Accounts** | MySQL `bi_datastore.billing_account` (`_DataSource = 'vBill'`) | `accountNumber`, `accountName`, `address`, `accountAttribute` |
| **Contacts** | Microsoft Dataverse (Dynamics) `contact` entity via FetchXML | `contact[]` under each account |

Both are joined on **AccountCode** (called `accountnumber` on the linked
Dataverse account record).

## Key transformations

1. **Unique account names** — `AccountName` is concatenated with `AccountCode`
   on every row (e.g. `John Doe 12345`). This guarantees uniqueness without
   needing a duplicate-detection pass.
2. **No blank contact fields** — `firstName`, `lastName`, and `EmailAddresses`
   that come back blank/null are substituted with safe defaults (`John`,
   `Doe`, `someone@example.com`) so the row still posts.
3. **Multi-value contact types** — `vgr_contacttypes` is a comma-separated
   list of Dataverse option codes. Each code is mapped to its OneBill label
   and emitted as an `associateValues` entry on the `Dynamics Contact Types`
   attribute, with `sequence` preserving order.
4. **Placeholder contact when empty** — accounts with zero Dynamics contacts
   get a single placeholder contact so the `contact[]` array is never empty.

## Execution order

| Section | What happens |
|---|---|
| 1 | Imports and config |
| 2 | Dataverse OAuth + FetchXML fetch |
| 3 | Contact-type code → label map |
| 4 | MySQL account query |
| 5 | Index contacts by account |
| 6 | OneBill token manager |
| 7 | Payload builder |
| 8 | POST worker |
| 9 | Parallel migration |
| 10 | Results, failures, error summary |

## 1. Imports and Configuration

All tunables and external endpoints live in one place. The expected `.env` keys are:

| Variable | Purpose |
|---|---|
| `CRM_TENANT_ID` / `CRM_CLIENT_ID` / `CRM_CLIENT_SECRET` | Azure AD app for Dataverse |
| `CRM_ENVIRONMENT_URL` | Dataverse env URL, no trailing slash |
| `DB_USERNAME` / `DB_PASSWORD` / `DB_HOST` | MySQL access |
| `CLIENT_ID` / `CLIENT_SECRET` / `API_USERNAME` / `API_PASSWORD` | OneBill OAuth |
| `PROXY_ACCOUNT_NUMBER` | OneBill proxy account header value |

`load_dotenv(override=True)` makes the `.env` authoritative over any pre-existing
shell env — important when re-running after rotating a secret.

In [22]:
# %pip install msal mysql-connector-python sqlalchemy python-dotenv requests pandas

import os
import json
import time
import logging
import threading
import urllib.parse
from datetime import datetime, timedelta
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd
from sqlalchemy import create_engine
from msal import ConfidentialClientApplication
from dotenv import load_dotenv

load_dotenv(override=True)

# --- Dataverse (Dynamics CRM) ---
CRM_TENANT_ID       = os.environ["CRM_TENANT_ID"]
CRM_CLIENT_ID       = os.environ["CRM_CLIENT_ID"]
CRM_CLIENT_SECRET   = os.environ["CRM_CLIENT_SECRET"]
CRM_ENVIRONMENT_URL = os.environ["CRM_ENVIRONMENT_URL"]

# --- MySQL ---
BI_DATASTORE_URL = (
    f"mysql+mysqlconnector://{os.environ['DB_USERNAME']}:{os.environ['DB_PASSWORD']}"
    f"@{os.environ['DB_HOST']}/bi_datastore"
)

# --- OneBill ---
ONEBILL_BASE_URL   = "https://sandbox-sg.onebillsoftware.com"
ONEBILL_TOKEN_URL  = f"{ONEBILL_BASE_URL}/oauth/token"
ONEBILL_PROXY_ACCT = os.environ["CREATION_PROXY_ACCOUNT_NUMBER"]

# --- Migration tunables ---
MAX_WORKERS        = 20
TOKEN_TTL_FALLBACK = 3500   # seconds; used only if OAuth response omits expires_in

# --- Defaults for blank fields ---
DEFAULT_FIRST_NAME = "John"
DEFAULT_LAST_NAME  = "Doe"
DEFAULT_EMAIL      = "someone@example.com"

# --- Logging ---
log_filename = f'migration_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(log_filename), logging.StreamHandler()],
)
logger = logging.getLogger(__name__)

2026-05-18 14:35:13,790 [WARNING] python-dotenv could not parse statement starting at line 1
2026-05-18 14:35:13,804 [WARNING] python-dotenv could not parse statement starting at line 5
2026-05-18 14:35:13,807 [WARNING] python-dotenv could not parse statement starting at line 11
2026-05-18 14:35:13,810 [WARNING] python-dotenv could not parse statement starting at line 15


## 2. Fetch Contacts from Dataverse

Pulls every active contact whose parent account has `accountcategorycode = 1`
and a non-null `accountnumber`, excluding contacts with `BILLING%` codes
(those represent legacy billing-only stubs, not real contacts).

### Pagination strategy

Dataverse caps each FetchXML response at 5,000 records. The official paging
mechanism (`paging-cookie`) can silently fail when combined with link-entity
joins — it sometimes returns page 1 forever. So we use **keyset pagination**
on `contactid` instead:

1. Order by `contactid` ascending (stable, unique, indexed).
2. First page: no extra filter.
3. Each subsequent page: inject `contactid gt <last seen>`.
4. Stop when a page returns fewer than 5,000 rows.

This is more robust than cookie-based paging and naturally resumable.

In [23]:
# {extra_condition} is filled in per-page with a `contactid gt <last_seen>` clause
FETCHXML_TEMPLATE = """
<fetch version="1.0" output-format="xml-platform" mapping="logical" no-lock="false" count="5000">
  <entity name="contact">
    <attribute name="fullname"/>
    <attribute name="emailaddress1"/>
    <attribute name="telephone1"/>
    <attribute name="contactid"/>
    <attribute name="vgr_contacttypes"/>
    <attribute name="mobilephone"/>
    <attribute name="firstname"/>
    <attribute name="lastname"/>
    <attribute name="vgr_contactcode"/>
    <order attribute="contactid" descending="false"/>
    <filter type="and">
        <condition attribute="parentcustomerid" operator="ne" value="7378af87-be17-eb11-a813-000d3a7940d5" uiname="Portal Default Account" uitype="account"/>
        {extra_condition}
    </filter>
    <link-entity name="account" from="accountid" to="parentcustomerid" link-type="inner" alias="AccountCode">
      <attribute name="accountnumber"/>
      <filter type="and">
        <condition attribute="vgr_datasource" operator="eq" value="vBill"/>
      </filter>
    </link-entity>
  </entity>
</fetch>
""".strip()

# The column the link-entity emits — we rename it to AccountCode after the fetch
LINKED_ACCOUNTNUMBER_COL = "AccountCode.accountnumber"


def get_dataverse_token() -> str:
    # OAuth2 client-credentials bearer token for the Dataverse environment.
    app = ConfidentialClientApplication(
        client_id=CRM_CLIENT_ID,
        client_credential=CRM_CLIENT_SECRET,
        authority=f"https://login.microsoftonline.com/{CRM_TENANT_ID}",
    )
    result = app.acquire_token_for_client(scopes=[f"{CRM_ENVIRONMENT_URL}/.default"])
    if "access_token" not in result:
        raise RuntimeError(f"Token acquisition failed: {result.get('error_description')}")
    return result["access_token"]


def get_contacts(token: str, max_pages: int = 50) -> pd.DataFrame:
    # Fetch all contacts from Dataverse using keyset pagination on contactid.
    headers = {
        "Authorization":    f"Bearer {token}",
        "OData-MaxVersion": "4.0",
        "OData-Version":    "4.0",
        "Accept":           "application/json",
        "Prefer":           "odata.maxpagesize=5000",
    }

    all_records: list[dict] = []
    last_contactid: str | None = None
    page = 1

    while True:
        if last_contactid is None:
            extra_condition = ""
        else:
            extra_condition = (
                f'<condition attribute="contactid" operator="gt" value="{last_contactid}"/>'
            )

        fetch = FETCHXML_TEMPLATE.format(extra_condition=extra_condition)
        url = f"{CRM_ENVIRONMENT_URL}/api/data/v9.2/contacts?fetchXml={urllib.parse.quote(fetch)}"

        response = requests.get(url, headers=headers, timeout=60)
        response.raise_for_status()
        records = response.json().get("value", [])

        if not records:
            break

        all_records.extend(records)
        new_last = records[-1]["contactid"]
        logger.info(
            f"Contacts page {page}: fetched {len(records):,} "
            f"(total so far: {len(all_records):,})"
        )

        if len(records) < 5000:
            break

        if new_last == last_contactid:
            logger.warning("contactid did not advance — stopping to avoid infinite loop")
            break

        last_contactid = new_last
        page += 1

        if page > max_pages:
            logger.warning(f"Hit max_pages safety limit ({max_pages})")
            break

    df = pd.DataFrame(all_records)
    logger.info(f"Done — {len(df):,} contacts loaded")
    return df

### Run the fetch and rename columns

`fillna` is applied here, **once**, so the rest of the pipeline never has to
guard against blank first/last/email — the substitution policy is enforced
at the boundary.

In [24]:
dataverse_token = get_dataverse_token()
df_contacts = get_contacts(dataverse_token)

# Drop OData noise, keep only the columns we use
df_contacts = df_contacts.drop(
    columns=[c for c in ["@odata.etag", "fullname"] if c in df_contacts.columns],
    errors="ignore",
)

df_contacts = df_contacts.rename(columns={
    "vgr_contactcode":         "ContactCode",
    "vgr_contacttypes":        "Dynamics_ContactTypes",
    "telephone1":              "PhoneWork",
    "mobilephone":             "PhoneMobile",
    "emailaddress1":           "EmailAddresses",
    "firstname":               "FirstName",
    "lastname":                "LastName",
    LINKED_ACCOUNTNUMBER_COL:  "AccountCode",
})

# Substitute defaults for blank fields (rule: no blank first/last/email allowed)
df_contacts["FirstName"]      = df_contacts["FirstName"].replace("", pd.NA).fillna(DEFAULT_FIRST_NAME)
df_contacts["LastName"]       = df_contacts["LastName"].replace("", pd.NA).fillna(DEFAULT_LAST_NAME)
df_contacts["EmailAddresses"] = df_contacts["EmailAddresses"].replace("", pd.NA).fillna(DEFAULT_EMAIL)

df_contacts['BillingContact'] = df_contacts['ContactCode'].str.startswith('BILLING')

df_contacts['OneBill_ContactType'] = df_contacts['ContactCode'].apply(
    lambda x: '0' if str(x).startswith('BILLING') else '1'
)


logger.info(f"Contacts after default substitution: {len(df_contacts):,}")
df_contacts.head()

2026-05-18 14:35:20,309 [INFO] Contacts page 1: fetched 5,000 (total so far: 5,000)
2026-05-18 14:35:26,392 [INFO] Contacts page 2: fetched 5,000 (total so far: 10,000)
2026-05-18 14:35:30,364 [INFO] Contacts page 3: fetched 5,000 (total so far: 15,000)
2026-05-18 14:35:33,846 [INFO] Contacts page 4: fetched 5,000 (total so far: 20,000)
2026-05-18 14:35:37,161 [INFO] Contacts page 5: fetched 5,000 (total so far: 25,000)
2026-05-18 14:35:41,955 [INFO] Contacts page 6: fetched 5,000 (total so far: 30,000)
2026-05-18 14:35:45,661 [INFO] Contacts page 7: fetched 5,000 (total so far: 35,000)
2026-05-18 14:35:50,036 [INFO] Contacts page 8: fetched 5,000 (total so far: 40,000)
2026-05-18 14:35:52,453 [INFO] Contacts page 9: fetched 1,844 (total so far: 41,844)
2026-05-18 14:35:52,510 [INFO] Done — 41,844 contacts loaded
2026-05-18 14:35:52,646 [INFO] Contacts after default substitution: 41,844


,PhoneMobile,contactid,LastName,ContactCode,FirstName,EmailAddresses,AccountCode,PhoneWork,Dynamics_ContactTypes,BillingContact,OneBill_ContactType
0,021569156,7266d549-cbfa-ee11-9f89-000d3a6a0933,Prentice,C-00090123,Lynn,lynn.prentice@gmail.com,47621153,NaN,NaN,False,1
1,0212141547,3b4848b4-7c5b-f011-bec1-000d3a6a2a4e,Sibbe,C-00100775,Judy,sibbe@actrix.co.nz,99993083,NaN,NaN,False,1
2,N/A,83b8ffe3-c69b-ef11-8a69-000d3a6a332a,Rose,C-00094944,Angel,accounts@techspanonline.com,99999381,6498276567,NaN,False,1
3,0210464741,0abb6f81-7d3d-ef11-a316-000d3a6a3623,Matthews,C-00092237,Angela,accounts@agedadvisor.co.nz,99937383,NaN,NaN,False,1
4,NaN,7b967bab-6c69-ef11-a670-000d3a6a3b2b,Clarke,C-00093561,Andre,andre.clarke@hbtech.co.nz,99993096,NaN,287790008,False,1


## 3. Dynamics Contact Type → OneBill Label Map

Dataverse stores `vgr_contacttypes` as a multi-select option-set. The API
returns it as a comma-separated list of numeric option codes:

    "287790000,287790001"   # two codes: Billing and Technical

We split on commas, strip whitespace, and map each code to the OneBill-side
label. Unknown codes are skipped silently rather than raised — a new
contact-type added in Dataverse before the map is updated won't break
the migration.

The **order** of the codes in the source string is preserved, which becomes
the `sequence` in the OneBill `associateValues` array.

In [25]:
CONTACT_TYPE_MAP = {
    "287790000": "Billing",
    "287790001": "Technical",
    "287790002": "Outage - Email",
    "287790009": "Outage - SMS",
    "287790003": "Primary",
    "287790004": "Technical - Data",
    "287790005": "Technical - Voice",
    "287790006": "Commercial",
    "287790008": "Communication",
    "287790007": "Voyager Staff",
}


def parse_contact_types(raw) -> list[str]:
    """Parse a vgr_contacttypes value into an ordered list of OneBill labels."""
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return []
    codes = [c.strip() for c in str(raw).split(",") if c.strip()]
    return [CONTACT_TYPE_MAP[c] for c in codes if c in CONTACT_TYPE_MAP]

## 4. MySQL Account Query

The query below is your simplified version, with two small additions baked
into the SQL so the Python side stays clean:

- **`AccountName_Cleaned`** strips legacy parenthetical markers (`(BOND)`,
  `(ICMS)`, `(Staff)`, `(X)`, `(In Liquidation)`, `(DECLINED)`,
  `(BOND - DECLINED)`) and the `zz-` prefix.
- **`AccountName_Unique`** appends the `AccountCode` to every cleaned name
  (e.g. `John Doe 12345`). This guarantees uniqueness on every row without
  needing a duplicate-detection pass — per your preference for the simplest
  rule.
- Address fields get safe defaults inline so the OneBill required-field
  validation never fails on a blank.

In [49]:
ACCOUNT_QUERY = """
SELECT
    `AccountName` AS `AccountName_Original`
    ,TRIM(
        REPLACE(
            REPLACE(
                REPLACE(
                    REPLACE(
                        REPLACE(
                            REPLACE(
                                REPLACE(
                                    REPLACE(
                                        REPLACE(
                                            REPLACE(`AccountName`, '(BOND - DECLINED)', ''),
                                        '(BOND)', ''),
                                    '(ICMS)', ''),
                                '(Staff)', ''),
                            '(X)', ''),
                        '(In Liquidation)', ''),
                    'zz-', ''),
                '(DECLINED)', ''),
            '(Operator)', ''),
        '(COMPRIMISED)', '') -- Even though it is spelt incorrectly there are no '(COMPROMISED)' accounts, only '(COMPRIMISED)', so we need to catch this too
    ) AS `AccountName_Cleaned`
    ,`AccountCode`
    ,`CreatedDate`
    ,`ClosedDate`
    ,`AccountType`
    ,CASE
		WHEN `AccountType` IN ('Residential', 'Actrix Residential', 'Consumer', 'Standard Account', 'Internal-Use Account', 'Staff')
        AND `AccountName` NOT LIKE '%Ltd%'
        AND `AccountName` NOT LIKE '%Limited%'
        AND `AccountName` NOT LIKE '%Pty%'
		THEN '1001' -- Individual Customer
        ELSE '1002' -- Business Customer 
	END AS `OneBill_AccountType`
    ,CASE
        WHEN `_temporary_crmonly_addr1` IS NULL OR `_temporary_crmonly_addr1` = '' THEN '1 Somewhere Place'
        ELSE `_temporary_crmonly_addr1`
    END AS `Address1`
    ,`_temporary_crmonly_addr2` AS `Address2`
    ,`_temporary_crmonly_suburb` AS `Suburb`
    ,CASE
        WHEN `_temporary_crmonly_city` IS NULL OR `_temporary_crmonly_city` = '' THEN 'Auckland'
        ELSE `_temporary_crmonly_city`
    END AS `City`
    ,CASE
        WHEN `_temporary_crmonly_postcode` IS NULL OR `_temporary_crmonly_postcode` = '' THEN '0001'
        ELSE `_temporary_crmonly_postcode`
    END AS `Postcode`
    ,`_temporary_crmonly_dob` AS `DateOfBirth`
FROM
    bi_datastore.billing_account
WHERE
    `_DataSource` = 'vBill'
ORDER BY
    `AccountCode` DESC
""".strip()

engine = create_engine(BI_DATASTORE_URL)
df_accounts = pd.read_sql(ACCOUNT_QUERY, con=engine)

# AccountCode as string for consistent dict lookups against Dataverse-side strings
df_accounts["AccountCode"] = df_accounts["AccountCode"].astype(str)

df_accounts['AccountName_Unique'] = df_accounts['AccountName_Cleaned'] + ' (' + df_accounts['AccountCode'] + ')'
df_accounts['AccountCode_Batch'] = df_accounts['AccountCode'] + '_' + os.environ["CREATION_PROXY_ACCOUNT_NUMBER"]

logger.info(f"Loaded {len(df_accounts):,} accounts from MySQL")
df_accounts.head()

2026-05-18 14:53:56,001 [INFO] Loaded 53,663 accounts from MySQL


,AccountName_Original,AccountName_Cleaned,AccountCode,CreatedDate,ClosedDate,AccountType,OneBill_AccountType,Address1,Address2,Suburb,City,Postcode,DateOfBirth,AccountName_Unique,AccountCode_Batch
0,Intagr8,Intagr8,Intagr8,2018-06-14,None,Residential,1001,"Level 4, 272 Parnell Road",None,Parnell,Auckland,1052,None,Intagr8 (Intagr8),Intagr8_32205
1,Michael Barnes (X),Michael Barnes,99999999,2018-06-14,None,Residential,1001,5A WYNDHAM STREET,None,ASHHURST,Auckland,4810,None,Michael Barnes (99999999),99999999_32205
2,Lincoln Barnes (X),Lincoln Barnes,99999998,2018-06-14,2025-05-23,Residential,1001,5/64 Dixon Street,None,Te Aro,Wellington,0001,None,Lincoln Barnes (99999998),99999998_32205
3,Victor Bi,Victor Bi,99999997,2018-06-14,None,Residential,1001,43 CHATHAM AVENUE\nPAREMOREMO\nNORTH SHORE\n0632,None,Paremoremo,Auckland,0001,None,Victor Bi (99999997),99999997_32205
4,Reta Hanson (X),Reta Hanson,99999996,2018-06-14,None,Residential,1001,225 Gloucester Road,None,Mt Maunganui,Auckland,0001,1980-05-07,Reta Hanson (99999996),99999996_32205


## 5. Index Contacts by AccountCode

Group all contact rows by their parent account's `AccountCode` so the
per-account lookup during the parallel POST loop is O(1).

`AccountCode` is cast to string on both sides — MySQL returns it as `int`
while Dataverse returns it as `str`, and `99961175 != "99961175"` would
silently produce empty contact lists.

In [40]:
def index_contacts_by_account(df: pd.DataFrame) -> dict[str, list[dict]]:
    """Group contact rows by AccountCode. Returns {AccountCode: [contact_dict, ...]}.

    Within each account, the BILLING- contact is placed FIRST. OneBill derives
    the account-level display name from the first contact in the payload, so
    putting the billing contact (ContactCode starts with 'BILLING') at index 0
    ensures the UI shows the right name. Stable sort preserves the original
    order among non-billing contacts (and among billing contacts if, for
    legacy-data reasons, there's more than one).
    """
    by_account: dict[str, list[dict]] = defaultdict(list)
    for _, row in df.iterrows():
        acct = row.get("AccountCode")
        if pd.isna(acct) or acct in (None, ""):
            continue
        by_account[str(acct)].append(row.to_dict())

    # Sort each account's contacts: BillingContact=True first, then everyone else.
    # `sorted` is stable, so original relative order is preserved within each bucket.
    for acct, contacts in by_account.items():
        contacts.sort(key=lambda c: not bool(c.get("BillingContact", False)))

    return dict(by_account)


contacts_by_account = index_contacts_by_account(df_contacts)
logger.info(
    f"Indexed {sum(len(v) for v in contacts_by_account.values()):,} contacts "
    f"across {len(contacts_by_account):,} accounts"
)

2026-05-18 14:46:19,250 [INFO] Indexed 41,836 contacts across 37,552 accounts


## 6. OneBill Token Manager (Thread-Safe)

One shared bearer token across all worker threads, refreshed proactively
~100 s before expiry. The lock is only held during refresh; the common
case (token still valid) returns under the lock with no I/O. If two threads
arrive while the token is stale, only one performs the refresh — the other
waits at the lock and then sees the freshly cached token.

In [41]:
class TokenManager:
    """Thread-safe bearer token cache with proactive refresh."""

    def __init__(self):
        self._lock = threading.Lock()
        self._token: str | None = None
        self._expires_at: datetime = datetime.min

    def get_token(self) -> str:
        with self._lock:
            if datetime.now() >= self._expires_at:
                self._refresh()
            return self._token

    def _refresh(self) -> None:
        logger.info("Refreshing OneBill OAuth token...")
        token_data = {
            "grant_type":    "password",
            "client_id":     os.environ["CLIENT_ID"],
            "client_secret": os.environ["CLIENT_SECRET"],
            "username":      os.environ["API_USERNAME"],
            "password":      os.environ["API_PASSWORD"],
        }
        response = requests.post(
            ONEBILL_TOKEN_URL,
            data=token_data,
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()
        self._token = payload["access_token"]
        ttl = payload.get("expires_in", TOKEN_TTL_FALLBACK)
        # Refresh 100 s early to absorb clock skew + in-flight requests
        self._expires_at = datetime.now() + timedelta(seconds=ttl - 100)
        logger.info("Token valid until %s", self._expires_at.strftime("%H:%M:%S"))


token_manager = TokenManager()

## 7. Payload Builder

Composes the JSON body matching the OneBill example payload exactly:

```
{
  "accountType": "1002",
  "accountName": "AccountName 12345",
  "address":          [ { ... } ],
  "contact":          [ { ... }, ... ],
  "accountAttribute": [ { "key": "vBill Account Types", "value": <AccountType> } ]
}
```

### Per-contact rules

- **`communicationPoint`** — emits `EMAIL` first, then `Phone` (mobile preferred,
  falling back to work phone). Null/blank values are skipped.
- **`contactAttributes`** — only emitted if the contact has at least one valid
  contact type. The `attributeValuesInfo.associateValues` array contains every
  parsed type in source order with `sequence` starting at 1.

### Placeholder contact

If an account has zero Dynamics contacts, a single placeholder contact is
inserted using the configured defaults so the `contact[]` array is never
empty.

In [42]:
def _clean(value):
    """Return None for blanks/NaN; pass everything else through unchanged."""
    if value is None:
        return None
    if isinstance(value, float) and pd.isna(value):
        return None
    if isinstance(value, str) and value.strip() == "":
        return None
    return value


def build_communication_points(contact: dict) -> list[dict]:
    """Emit Email + Phone communication points, skipping blanks. Mobile preferred over work phone."""
    points: list[dict] = []

    email = _clean(contact.get("EmailAddresses"))
    if email is not None:
        points.append({"type": "EMAIL", "value": str(email)})

    # Prefer mobile, fall back to work phone (NaN-safe)
    phone = _clean(contact.get("PhoneMobile")) or _clean(contact.get("PhoneWork"))
    if phone is not None:
        points.append({"type": "Phone", "value": str(phone)})

    return points


def build_contact_block(contact: dict) -> dict:
    """Build a single OneBill contact entry from a Dynamics contact row."""
    block = {
        "firstName":          contact["FirstName"],
        "lastName":           contact["LastName"],
        'contactType':        contact.get("OneBill_ContactType"),
        'primaryContact':     contact.get("BillingContact", False),
        'billingContact':     contact.get("BillingContact", False),
        "communicationPoint": build_communication_points(contact),
    }

    types = parse_contact_types(contact.get("Dynamics_ContactTypes"))
    if types:
        block["contactAttributes"] = [
            {
                "key":                   "Dynamics Contact Types",
                "value":                 types[0],
                "multipleEntriesConfig": "ENABLED",
                "attributeValuesInfo": {
                    "associateValues": [
                        {"value": t, "sequence": i + 1}
                        for i, t in enumerate(types)
                    ],
                },
            },
            {
                "key": "Contact Code",
                "value": contact["ContactCode"]
            }
        ]

    return block


def build_placeholder_contact() -> dict:
    """A single safe contact used when an account has zero Dynamics contacts."""
    return {
        "firstName": DEFAULT_FIRST_NAME,
        "lastName":  DEFAULT_LAST_NAME,
        "communicationPoint": [
            {"type": "EMAIL", "value": DEFAULT_EMAIL},
        ],
    }


def build_account_payload(row: pd.Series, contacts_by_account: dict[str, list[dict]]) -> str:
    """Build the OneBill account-creation payload for a single MySQL account row."""
    # Convert pandas row -> dict, then NaN -> None everywhere (json.dumps emits
    # 'NaN' for float NaN which OneBill won't accept as valid JSON)
    row = {k: _clean(v) if not isinstance(v, (list, dict)) else v for k, v in row.to_dict().items()}

    # --- Resolve contacts for this account
    extra_contacts = contacts_by_account.get(str(row["AccountCode"]), [])
    contact_list = [build_contact_block(c) for c in extra_contacts]
    if not contact_list:
        contact_list = [build_placeholder_contact()]
    
    def serialize_date(value, fmt=None):
        if value is None:
            return None
        if hasattr(value, 'isoformat'):
            return value.strftime(fmt) if fmt else value.isoformat()
        if fmt:
            try:
                return datetime.strptime(str(value), '%Y-%m-%d').strftime(fmt)
            except ValueError:
                return str(value)
        return str(value)

    # --- Address (single element list as per the example payload)
    address_block = {
        "addLine1":        row["Address1"],
        "addLine2":        row.get("Address2"),
        "city":            row["City"],
        "state":           None,
        "country":         "New Zealand",
        "zip":             str(row["Postcode"]) if row.get("Postcode") is not None else None,
        "defaultShipping": True,
        "defaultBilling":  True,
    }

    payload = {
        "accountType":   row['OneBill_AccountType'],
        "accountName":   row["AccountName_Cleaned"],
        'accountNumber': row["AccountCode_Batch"],
        'accountingDisplayName': row["AccountName_Unique"],
        'activationStartDate': row["CreatedDate"].strftime("%Y-%m-%d"),
        "address":       [address_block],
        "contact":       contact_list,
        "accountAttribute": [
        {
            "key":   "vBill Account Types",
            "value": row.get("AccountType"),
        },
        {
            'key': 'Date Of Birth',
            'value': serialize_date(row.get("DateOfBirth"), "%d/%m/%Y")
        },
        {
            'key': 'vBill Account Name',
            'value': row.get("AccountName_Original")
        }]
    }

    return json.dumps(payload)

## 8. POST to OneBill + Per-Row Worker

`create_onebill_account` performs the actual POST and treats
`validationResponse.successful = False` as a failure even when the HTTP
status is 200 — OneBill returns business-logic errors that way.

`migrate_row` is the per-account worker. Build time and network time are
recorded separately so any slowdown is easy to attribute from the results
DataFrame.

In [43]:
def create_onebill_account(session: requests.Session, base_url: str, payload: str) -> dict:
    """POST one account to OneBill. Raises ValueError on validation failure."""
    url = f"{base_url}/rest/SubscriberService/v1/subscriber"
    headers = {"Authorization": f"Bearer {token_manager.get_token()}"}

    response = session.post(url, headers=headers, data=payload, timeout=30)
    response.raise_for_status()
    data = response.json()

    validation = data.get("validationResponse", {})
    if not validation.get("successful", True):
        errors   = validation.get("validationErrorInfo", [])
        messages = "; ".join(e.get("message", "") for e in errors)
        raise ValueError(messages or "validationResponse.successful = false")

    return data


def migrate_row(
    row: pd.Series,
    session: requests.Session,
    contacts_by_account: dict[str, list[dict]],
) -> dict:
    """Build + POST a single account, returning a result dict for the summary."""
    account_code = row["AccountCode"]
    account_name = row["AccountName_Unique"]

    t0      = time.perf_counter()
    payload = build_account_payload(row, contacts_by_account)
    t_build = time.perf_counter() - t0

    t_net = 0.0
    status = "failed"
    error  = None
    onebill_id = None

    try:
        t1 = time.perf_counter()
        response = create_onebill_account(session, ONEBILL_BASE_URL, payload)
        t_net = time.perf_counter() - t1

        onebill_id = response.get("accountId", "unknown")
        status = "success"
        logger.info(
            f"  [OK] {account_code} (OneBill id={onebill_id}) — "
            f"build={t_build*1000:.0f}ms net={t_net*1000:.0f}ms"
        )

    except Exception as e:
        if "t1" in locals():
            t_net = time.perf_counter() - t1
        error = str(e)
        logger.error(f"  [FAIL] {account_code} — {error}")

    return {
        "AccountCode":       account_code,
        "AccountName":       account_name,
        "status":            status,
        "onebill_id":        onebill_id,
        "error":             error,
        "elapsed_build_ms":  round(t_build * 1000, 1),
        "elapsed_net_ms":    round(t_net   * 1000, 1),
    }

## 9. Parallel Migration Loop

Fans the account list out across `MAX_WORKERS` threads. The HTTP session
is shared (with a pool sized to match worker count) so connections are
reused rather than re-established for every request.

OneBill's sandbox has been observed to throttle past ~20 concurrent
writers — increase `MAX_WORKERS` cautiously.

In [44]:
def migrate(
    df: pd.DataFrame,
    contacts_by_account: dict[str, list[dict]],
    max_workers: int = MAX_WORKERS,
) -> pd.DataFrame:
    """Migrate every account in df to OneBill in parallel."""
    session = requests.Session()
    adapter = requests.adapters.HTTPAdapter(
        pool_connections=max_workers,
        pool_maxsize=max_workers,
    )
    session.mount("https://", adapter)
    session.headers.update({
        "proxy_accountNumber": ONEBILL_PROXY_ACCT,
        "Content-Type":        "application/json",
    })

    rows  = [row for _, row in df.iterrows()]
    total = len(rows)
    results: list[dict] = []

    logger.info(f"Starting migration of {total:,} accounts with {max_workers} workers...")
    wall_start = time.perf_counter()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(migrate_row, row, session, contacts_by_account): row["AccountCode"]
            for row in rows
        }

        for i, future in enumerate(as_completed(futures), start=1):
            results.append(future.result())

            if i % 50 == 0 or i == total:
                ok   = sum(1 for r in results if r["status"] == "success")
                fail = sum(1 for r in results if r["status"] == "failed")
                logger.info(f"Progress: {i}/{total} — {ok} ok, {fail} failed")

    wall_elapsed = time.perf_counter() - wall_start
    results_df = pd.DataFrame(results)
    success = (results_df["status"] == "success").sum()
    failed  = (results_df["status"] == "failed").sum()

    logger.info(
        f"Migration done in {wall_elapsed:.1f}s — "
        f"{success} succeeded, {failed} failed. (log: {log_filename})"
    )

    # --- Profiling summary
    print("\n=== Profiling Summary ===")
    print(f"Total wall time:          {wall_elapsed:.1f}s")
    if wall_elapsed > 0:
        print(f"Throughput:               {total / wall_elapsed:.1f} accounts/s")
    print(f"Avg build time per row:   {results_df['elapsed_build_ms'].mean():.1f}ms")
    print(f"Avg network time per row: {results_df['elapsed_net_ms'].mean():.1f}ms")
    print(f"Max network time:         {results_df['elapsed_net_ms'].max():.1f}ms")
    print(f"P95 network time:         {results_df['elapsed_net_ms'].quantile(0.95):.1f}ms")
    print("=========================")

    return results_df

## 10. Run the Migration

Kicks off the full parallel run.

In [ ]:
results_df = migrate(df_accounts, contacts_by_account)

failures = results_df[results_df["status"] == "failed"]
print(f"\nFailed rows ({len(failures):,}):")
failures.head(20)

2026-05-18 14:54:14,408 [INFO] Starting migration of 53,663 accounts with 20 workers...
2026-05-18 14:54:44,075 [ERROR]   [FAIL] 99999996 — Account number 99999996_32205 already exists.
2026-05-18 14:54:44,212 [ERROR]   [FAIL] 99999980 — Account number 99999980_32205 already exists.
2026-05-18 14:54:44,368 [ERROR]   [FAIL] 99999985 — Account number 99999985_32205 already exists.
2026-05-18 14:54:44,369 [ERROR]   [FAIL] 99999979 — Account number 99999979_32205 already exists.
2026-05-18 14:54:44,403 [ERROR]   [FAIL] 99999982 — Account number 99999982_32205 already exists.
2026-05-18 14:54:44,598 [ERROR]   [FAIL] 99999997 — Account number 99999997_32205 already exists.
2026-05-18 14:54:44,599 [ERROR]   [FAIL] 99999978 — Account number 99999978_32205 already exists.
2026-05-18 14:54:44,633 [ERROR]   [FAIL] 99999976 — Account number 99999976_32205 already exists.
2026-05-18 14:54:44,692 [ERROR]   [FAIL] 99999977 — Account number 99999977_32205 already exists.
2026-05-18 14:54:44,924 [ERROR

### Export failures

In [46]:
out_path = f'Failed_Migrations_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
failures.to_csv(out_path, index=False)
print(f"Wrote {len(failures):,} failures to {out_path}")

Wrote 2 failures to Failed_Migrations_20260518_144655.csv


### Error summary

Groups failures by error message so you can triage the biggest cluster
first, re-run, and repeat.

In [47]:
if not failures.empty:
    error_summary = (
        failures.groupby("error")
        .agg(count=("AccountCode", "size"),
             example_account=("AccountCode", "first"))
        .sort_values("count", ascending=False)
        .reset_index()
    )
    print(f"Distinct error messages: {len(error_summary):,}")
else:
    print("No failures to summarise.")
    error_summary = pd.DataFrame()
error_summary

Distinct error messages: 2


,error,count,example_account
0,Account number 99999999_32205 already exists.,1,99999999
1,Account number Intagr8_32205 already exists.,1,Intagr8


### Failed accounts → associated contacts

Joins each failed account back to its Dynamics contacts so you can inspect
them side-by-side with the failure reason. Useful for spotting patterns
like a bad email format across all of an account's contacts, or a
`vgr_contacttypes` value that didn't map to anything in `CONTACT_TYPE_MAP`.

The join is a LEFT join — failed accounts with zero Dynamics contacts are
kept (with contact columns null) so they aren't silently dropped.

In [48]:
def lookup_failed_contacts(failures_df: pd.DataFrame, df_contacts: pd.DataFrame) -> pd.DataFrame:
    """Join failed accounts to their Dynamics contacts (long-format)."""
    if failures_df.empty:
        return pd.DataFrame()

    contacts_lookup = df_contacts.copy()
    contacts_lookup["AccountCode"] = contacts_lookup["AccountCode"].astype(str)

    fails = failures_df.copy()
    fails["AccountCode"] = fails["AccountCode"].astype(str)

    joined = fails.merge(
        contacts_lookup[[
            "AccountCode", "FirstName", "LastName",
            "EmailAddresses", "PhoneMobile", "PhoneWork", "Dynamics_ContactTypes",
        ]],
        on="AccountCode",
        how="left",
        suffixes=("_account", "_contact"),
    )

    return joined


failures_with_contacts = lookup_failed_contacts(failures, df_contacts)

if not failures_with_contacts.empty:
    fwc_path = f'Failed_Migrations_with_Contacts_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    failures_with_contacts.to_csv(fwc_path, index=False)
    print(f"Wrote {len(failures_with_contacts):,} failure+contact rows to {fwc_path}")

failures_with_contacts.head(20)

Wrote 5 failure+contact rows to Failed_Migrations_with_Contacts_20260518_144655.csv


,AccountCode,AccountName,status,onebill_id,error,elapsed_build_ms,elapsed_net_ms,FirstName,LastName,EmailAddresses,PhoneMobile,PhoneWork,Dynamics_ContactTypes
0,Intagr8,Intagr8 (Intagr8),failed,None,Account number Intagr8_32205 already exists.,0.3,32362.8,Intagr8,Intagr8,someone@example.com,+64211234567,+6494444444,"287790000,287790003"
1,99999999,Michael Barnes (99999999),failed,None,Account number 99999999_32205 already exists.,0.4,34333.6,Jazel,Caasi,jazelmae@gmail.com,0221097073,NaN,287790006
2,99999999,Michael Barnes (99999999),failed,None,Account number 99999999_32205 already exists.,0.4,34333.6,John,Michael Barnes,sikspinner@gmail.com,+64204097397,NaN,"287790000,287790003"
3,99999999,Michael Barnes (99999999),failed,None,Account number 99999999_32205 already exists.,0.4,34333.6,Ning,LI,nzstter@gmail.com,0221231657,NaN,287790006
4,99999999,Michael Barnes (99999999),failed,None,Account number 99999999_32205 already exists.,0.4,34333.6,Hannah,Barnes,Hannahnbarnes@gmail.com,0211607985,NaN,287790006
